# Itinerary Generator Pipeline — Step-by-Step

Walks through `services/trip-generation/` one phase at a time. Run cells
top-to-bottom the first time; afterwards you can tweak any intermediate
variable and re-run just the downstream phase.

**Phases**
1. Route search & composition
2. POI retrieval + enrichment + union with route POIs
3. LLM pick (Gemini)
4. Solver — order stops, enforce opening hours & meal anchors
5. Persist (optional — writes to Supabase)

**Before you start:** see `notebooks/README.md` for tslab install. Launch
`jupyter lab` from the **repo root**, not from `notebooks/`.

## Setup — load `.env.local` and sanity-check keys

In [ ]:
import { dirname, resolve } from "node:path";
import { existsSync } from "node:fs";

let repoRoot = process.cwd();
while (!existsSync(resolve(repoRoot, ".env.local")) && dirname(repoRoot) !== repoRoot) {
  repoRoot = dirname(repoRoot);
}

const envPath = resolve(repoRoot, ".env.local");
if (!existsSync(envPath)) {
  throw new Error(`Missing .env.local above ${process.cwd()}. Did you start jupyter inside this repo?`);
}

if (process.cwd() !== repoRoot) {
  process.chdir(repoRoot);
}
import "tsx/cjs";
process.loadEnvFile(envPath);

const required = [
  "NEXT_PUBLIC_SUPABASE_URL",
  "SUPABASE_SECRET_KEY",
  "GEMINI_API_KEY",
];
const missing = required.filter((k) => !process.env[k]);
if (missing.length) throw new Error(`Missing env vars: ${missing.join(", ")}`);

console.log("repo root:", repoRoot);
console.log("env loaded:", required.map((k) => `${k}=${process.env[k]?.slice(0, 8)}...`).join("  "));
console.log("GOOGLE_PLACES_API_KEY:", process.env.GOOGLE_PLACES_API_KEY ? "set" : "(missing - enrichment will be skipped)");


repo root: E:\Projects\travel-sync-ai


env loaded: NEXT_PUBLIC_SUPABASE_URL=https://...  SUPABASE_SECRET_KEY=sb_secre...  GEMINI_API_KEY=AIzaSyCx...


GOOGLE_PLACES_API_KEY: set


## Imports — pipeline functions

Path alias `@/*` is wired through `notebooks/tsconfig.json` so we can use
the same imports the production code does.

In [ ]:
const routeEngine = require("@/services/trip-generation/route-engine");
const poiEngine = require("@/services/trip-generation/poi-engine");
const solver = require("@/services/trip-generation/solver");
const orchestrator = require("@/services/trip-generation/orchestrator");
const { randomBytes } = require("node:crypto");

Object.assign(globalThis, {
  searchRoutesByVibe: routeEngine.searchRoutesByVibe,
  composeFromRoutes: routeEngine.composeFromRoutes,
  searchPoisByVibe: poiEngine.searchPoisByVibe,
  loadPoisByIds: poiEngine.loadPoisByIds,
  enrichWithLiveData: poiEngine.enrichWithLiveData,
  solveItinerary: solver.solveItinerary,
  PACE_CAPS: solver.PACE_CAPS,
  __notebook: orchestrator.__notebook,
});

const genId = randomBytes(4).toString("hex");
(globalThis as any).genId = genId;
console.log("genId for this notebook run:", genId);


genId for this notebook run: 932bfe56


## Phase 0 — Survey input

Edit this cell to change the destination, party, vibe, etc., then re-run
everything below.

In [ ]:
const answers = {
  destination: "Kyoto",
  duration_days: 3,
  party: "couple",
  party_size: 2,
  budget_tier: "mid",
  vibe: ["culture", "foodie"],
  pace: "balanced",
  must_haves: null,
};

const input = {
  answers,
  authorLineUserId: "U_notebook_dev",
  startDate: undefined, // defaults to today + 14d
};

__notebook.validateAnswers(input);
const startWeekday = __notebook.deriveStartWeekday(input.startDate);
console.log("validated. startWeekday =", startWeekday, " (0=Sun … 6=Sat)");


validated. startWeekday = 2  (0=Sun … 6=Sat)


## Phase 1a — Search curated routes

Vector-search `route_templates` by vibe. Returns up to 10 candidate
single-day routes, scored on similarity + boost + quality + pinned vibes.
Gate is 0.72 — anything below is filtered out.

In [ ]:
const routes = await searchRoutesByVibe({
  destination: answers.destination!,
  vibe: answers.vibe,
  pace: answers.pace,
  budget: answers.budget_tier,
  k: 10,
  genId,
});

console.log(`routes found: ${routes.length}`);
console.table(routes.slice(0, 10).map((r) => ({
  routeId: r.routeId.slice(0, 8),
  title: r.title,
  score: r.finalScore.toFixed(3),
  similarity: r.similarity.toFixed(3),
  places: r.placeIds.length,
})));


{"level":"info","msg":"[route-engine] no rows from RPC","ts":"2026-05-26T03:24:51.041Z","genId":"932bfe56","destination":"Kyoto","pace":"balanced"}


routes found: 0


┌─────────┐
│ (index) │
├─────────┤
└─────────┘


## Phase 1b — Compose routes into day-slots

Greedy packer assigns routes to days, avoiding place_id collisions. Any
day that isn't covered drops into `uncoveredDays`, which Phase 3 (LLM)
will fill.

In [ ]:
const compose = composeFromRoutes(routes, answers.duration_days!);

console.log("covered days:");
for (const [day, route] of compose.coveredDays) {
  console.log(`  D${day}  ${route.title}  (${route.placeIds.length} stops)`);
}
console.log("uncovered days (LLM will pick for these):", compose.uncoveredDays);
console.log("place_ids reserved by routes:", compose.usedPlaceIds.size);


covered days:


uncovered days (LLM will pick for these): [ 1, 2, 3 ]


place_ids reserved by routes: 0


## Phase 2a — Retrieve POI candidates

Vector ANN search on `poi_embeddings`. Falls back to Google Places text
search if the corpus is cold for this destination.

In [ ]:
const poiCandidates = await searchPoisByVibe({
  destination: answers.destination!,
  vibe: answers.vibe,
  pace: answers.pace,
  budget: answers.budget_tier,
  k: 30,
  genId,
});

console.log(`POI candidates: ${poiCandidates.length}`);
console.table(poiCandidates.slice(0, 15).map((p) => ({
  name: p.name,
  type: p.itemType,
  similarity: p.similarity.toFixed(3),
  tags: (p.tags ?? []).slice(0, 4).join(","),
})));


{"level":"warn","msg":"[poi-engine] vector search returned 0 rows, falling back","ts":"2026-05-26T03:24:51.475Z","genId":"932bfe56","destination":"Kyoto"}


{"level":"info","msg":"[poi-engine] live text fallback","ts":"2026-05-26T03:24:52.562Z","genId":"932bfe56","destination":"Kyoto","buckets":"activity,restaurant,hotel","returned":30}


POI candidates: 30


┌─────────┬─────────────────────────────────────────────────────────────┬──────────────┬────────────┬──────┐
│ (index) │ name                                                        │ type         │ similarity │ tags │
├─────────┼─────────────────────────────────────────────────────────────┼──────────────┼────────────┼──────┤
│ 0       │ 'Kinkaku-ji'                                                │ 'activity'   │ '0.500'    │ ''   │
│ 1       │ '伏見稻荷大社'                                              │ 'activity'   │ '0.500'    │ ''   │
│ 2       │ '二條城'                                                    │ 'activity'   │ '0.500'    │ ''   │
│ 3       │ 'SAMURAI NINJA MUSEUM Kyoto'                                │ 'activity'   │ '0.500'    │ ''   │
│ 4       │ '清水寺'                                                    │ 'activity'   │ '0.500'    │ ''   │
│ 5       │ '慈照寺'                                                    │ 'activity'   │ '0.500'    │ ''   │
│ 6       │ '嵐山'                  

## Phase 2b — Union with route POIs

Routes lock specific place_ids; we materialize them as POI candidates
and merge into the shortlist. Route POIs take precedence.

In [ ]:
const routePois = await loadPoisByIds(Array.from(compose.usedPlaceIds), genId);
const allCandidates = __notebook.unionByPlaceId(routePois, poiCandidates);

console.log(`route POIs: ${routePois.length} | search POIs: ${poiCandidates.length} | unioned: ${allCandidates.length}`);
if (allCandidates.length === 0) throw new Error("no candidates — pipeline would abort with no_candidates");


route POIs: 0 | search POIs: 30 | unioned: 30


## Phase 2c — Enrich with live data (Google Places)

Batch-fetches address, coords, and opening periods. The solver requires
coords + opening hours; without them, a POI is effectively unschedulable.

In [ ]:
const enriched = await enrichWithLiveData(allCandidates);

const withCoords = enriched.filter((e) => e.lat != null && e.lng != null).length;
const withHours = enriched.filter((e) => (e.live?.openingPeriods?.length ?? 0) > 0).length;
console.log(`enriched: ${enriched.length} total | ${withCoords} with coords | ${withHours} with opening hours`);
console.table(enriched.slice(0, 10).map((e) => ({
  name: e.name,
  type: e.itemType,
  coords: e.lat != null ? `${e.lat.toFixed(3)},${e.lng!.toFixed(3)}` : "(none)",
  hours: e.live?.openingPeriods?.length ?? 0,
  address: e.live?.address?.slice(0, 40) ?? "(none)",
})));


enriched: 30 total | 30 with coords | 20 with opening hours


┌─────────┬──────────────────────────────┬────────────┬──────────────────┬───────┬─────────────────────────────────────────────┐
│ (index) │ name                         │ type       │ coords           │ hours │ address                                     │
├─────────┼──────────────────────────────┼────────────┼──────────────────┼───────┼─────────────────────────────────────────────┤
│ 0       │ 'Kinkaku-ji'                 │ 'activity' │ '35.039,135.729' │ 7     │ '1 Kinkakujichō, Kita Ward, Kyoto, 603-83'  │
│ 1       │ '伏見稻荷大社'               │ 'activity' │ '34.968,135.779' │ 1     │ '68 Fukakusa Yabunouchichō, Fushimi Ward,'  │
│ 2       │ '二條城'                     │ 'activity' │ '35.014,135.748' │ 7     │ '541 Nijōjōchō, Nakagyo Ward, Kyoto, 604-'  │
│ 3       │ 'SAMURAI NINJA MUSEUM Kyoto' │ 'activity' │ '35.007,135.764' │ 7     │ '109 Horinouechō, Nakagyo Ward, Kyoto, 60'  │
│ 4       │ '清水寺'                     │ 'activity' │ '34.995,135.785' │ 7     │ '1-chōme-294 Kiyomizu, Hig

## Phase 3 — LLM pick (Gemini)

Gemini sees only place_id, name, type, tags, and a short summary — never
coords or hours. It returns `{ title, summary, tags, days: [{day_number,
place_ids[]}] }`. The orchestrator filters hallucinated/duplicate IDs and
truncates to the pace cap before returning.

We only ask the LLM to cover days that routes didn't already claim.

In [ ]:
let pick = null;

if (compose.uncoveredDays.length === 0) {
  console.log("all days route-covered — skipping LLM. Synthesizing title/summary from routes.");
  pick = __notebook.synthesizePickFromRoutes(compose, answers.destination!);
} else {
  pick = await __notebook.llmPickAssignment(
    input,
    enriched,
    [], // no prior infeasibility issues on first attempt
    undefined, // no prior pick
    {
      onlyDays: compose.uncoveredDays,
      excludePlaceIds: compose.usedPlaceIds,
      genId,
      attempt: 0,
    }
  );
}

console.log("title  :", pick.title);
console.log("summary:", pick.summary);
console.log("tags   :", pick.tags.join(", "));
const byId = new Map(enriched.map((p) => [p.placeId, p]));
for (const d of pick.days) {
  const names = d.place_ids.map((id) => byId.get(id)?.name ?? `(unknown ${id.slice(0,8)})`);
  console.log(`  D${d.day_number}: ${names.join(" → ")}`);
}


{"level":"info","msg":"[gen] phase 3: llm input","ts":"2026-05-26T03:24:53.028Z","genId":"932bfe56","attempt":0,"onlyDays":"1,2,3","shortlistSize":30,"shortlistByType":"activity=10,restaurant=10,hotel=10","repairIssues":"(none)","hadPrior":false}


[gemini] calling gemini-2.5-flash with JSON output...


[gemini] raw response length: 902


title  : 京都浪漫文化美食三日遊


summary: 為期三天的京都行程，專為情侶設計，預算中等，節奏平衡。此行程涵蓋了京都最具代表性的文化遺產、自然風光，並安排了多樣化的美食體驗，讓您深入感受古都的魅力。


tags   : 文化, 美食, 情侶, 平衡, 京都


  D1: 伏見稻荷大社 → 清水寺 → Kyoto Tempura Ten no Meshi Gionhonten → Gion Loka


  D2: 嵐山 → 嵐山竹林小徑 → 嵐山猴子公園 → Kobe Beef Kisshokichi - Kyoto Main Store


  D3: Kinkaku-ji → 二條城 → SAMURAI NINJA MUSEUM Kyoto → 御好燒 克(Katsu) → 京都塔


### (Optional) Hand-edit the LLM pick

If you want to override the assignment before the solver runs, mutate
`pick.days` here. Example: force a different place_id on day 2.

```ts
// pick!.days = pick!.days.map(d => d.day_number === 2
//   ? { ...d, place_ids: [enriched[0].placeId, enriched[3].placeId] }
//   : d
// );
```

## Phase 4 — Solver

Brute-force permutes each day's stops (≤6 → ≤720 permutations), simulates
travel via Haversine, enforces opening hours and meal anchors (lunch
12–14, dinner 18–20). Returns either `feasible` with timed stops or
`infeasible` with issues — in the real pipeline, infeasibility triggers a
repair loop (LLM swap + route demotion, ≤2 attempts).

In [ ]:
const solved = __notebook.trySolve(pick, enriched, input, startWeekday, compose);

if (solved.kind === "feasible") {
  console.log("FEASIBLE\n");
  for (const day of solved.days) {
    console.log(`Day ${day.dayNumber}`);
    for (const s of day.stops) {
      const arrive = `${String(Math.floor(s.arriveMinutes/60)).padStart(2,"0")}:${String(s.arriveMinutes%60).padStart(2,"0")}`;
      const depart = `${String(Math.floor(s.departMinutes/60)).padStart(2,"0")}:${String(s.departMinutes%60).padStart(2,"0")}`;
      console.log(`  ${arrive}–${depart}  ${s.poi.name}  [${s.poi.itemType}]`);
    }
    console.log("");
  }
} else {
  console.log("INFEASIBLE — issues:");
  for (const issue of solved.issues) {
    console.log(`  D${issue.dayNumber} / ${issue.reason}: ${issue.detail}`);
    if (issue.offendingPlaceIds.length) {
      const names = issue.offendingPlaceIds.map((id) => byId.get(id)?.name ?? id.slice(0,8));
      console.log(`    offending: ${names.join(", ")}`);
    }
  }
  console.log("\nTo simulate the repair loop, re-run Phase 3 with these issues passed in as the 3rd arg of llmPickAssignment, and `pick` as the 4th arg.");
}


INFEASIBLE — issues:


  D1 / no_valid_permutation: No ordering satisfies opening hours and travel windows


    offending: 伏見稻荷大社, 清水寺, Kyoto Tempura Ten no Meshi Gionhonten, Gion Loka


  D2 / no_valid_permutation: No ordering satisfies opening hours and travel windows


    offending: 嵐山, 嵐山竹林小徑, 嵐山猴子公園, Kobe Beef Kisshokichi - Kyoto Main Store


  D3 / no_valid_permutation: No ordering satisfies opening hours and travel windows


    offending: Kinkaku-ji, 二條城, SAMURAI NINJA MUSEUM Kyoto, 御好燒 克(Katsu), 京都塔



To simulate the repair loop, re-run Phase 3 with these issues passed in as the 3rd arg of llmPickAssignment, and `pick` as the 4th arg.


### (Optional) Manual repair attempt

Uncomment to re-ask the LLM with the infeasibility report. The orchestrator
does this automatically up to `MAX_REPAIR_ATTEMPTS = 2` times.

```ts
// if (solved.kind === "infeasible" && pick) {
//   const repairedPick = await __notebook.llmPickAssignment(
//     input,
//     enriched,
//     solved.issues.filter((i) => !compose.coveredDays.has(i.dayNumber)),
//     pick,
//     { onlyDays: compose.uncoveredDays, excludePlaceIds: compose.usedPlaceIds, genId, attempt: 1 }
//   );
//   const resolved = __notebook.trySolve(repairedPick, enriched, input, startWeekday, compose);
//   console.log("after repair:", resolved.kind);
// }
```

## Phase 5 — Persist (commented out)

Writes a real `trip_templates` + `trip_template_versions` +
`trip_template_items` row to Supabase. **Uncomment only when you want a
real artifact.**

In [ ]:
// if (solved.kind === "feasible" && pick) {
//   const out = await __notebook.persistTemplate(input, pick, solved.days, enriched, compose);
//   console.log("persisted:", out);
// } else {
//   console.log("skipping persist — solver was not feasible.");
// }
